# Практическая работа №1. Часть 2Г
## Обратное распространение и autograd

**Цель:** понять, как нейросеть находит вклад каждого веса в ошибку; написать свой маленький механизм автоматического дифференцирования (класс `Value`), проверить градиенты PyTorch численно и написать цикл обучения, который будем использовать во всех следующих работах.

> **Правило части.** В заданиях Г1–Г3 внутри своих функций – только Python и модуль `math`; `torch` используется лишь в готовых ячейках сверки. Начиная с Г4 – `torch`, включая `torch.relu`, autograd (`.backward()`) и `torch.optim`.

**Задача:** два сквозных примера со слайдов. Первый – одно выражение $L = (w x + b - y)^2$ при $x = 2$, $w = 3$, $b = 1$, $y = 5$. Второй – сеть XOR из части 1С, в которой «испорчен» один вес: $w = (1; -1{,}5)$ вместо $(1; -2)$. Мы найдем, как нужно поправить веса, и обучим сеть. В конце – обучение по мини-пакетам на данных «две луны».

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию ниже.

## Как сеть узнает, какой вес виноват

**1. Зачем нужен градиент.** Обучение – это подбор весов, при которых потеря $L$ минимальна. В части 4 практической работы 9 класса логистическую регрессию учили **градиентным спуском**: вес сдвигают маленьким шагом против производной потери

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}.$$

Знак производной показывает, в какую сторону двигать вес, величина – насколько сильно потеря от него зависит («насколько вес виноват»), $\eta$ – шаг (скорость обучения, learning rate). Для логистической регрессии формулу производной выводили вручную. У сети $101\,770$ весов и формула сложная – нужен **общий способ**, который посчитает все производные сразу.

*Наивный способ* – пошевелить каждый вес: $\dfrac{\partial L}{\partial w} \approx \dfrac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$. Это два прогона сети на каждый вес, для сети 784 → 128 → 10 – $203\,540$ прогонов на **один** шаг обучения. Слишком дорого; такой способ используют только для проверки (задание Г4).

**2. Граф вычислений.** Любую формулу можно разложить на простые операции. Для $L = (w x + b - y)^2$:

$$m = w \cdot x, \qquad z = m + b, \qquad d = z - y, \qquad L = d^2.$$

Это **граф вычислений**: узлы – операции, стрелки – числа, которые передаются от операции к операции. **Прямой проход** считает значения от входов к $L$ и запоминает их: при $x = 2$, $w = 3$, $b = 1$, $y = 5$ получаем $m = 6$, $z = 7$, $d = 2$, $L = 4$.

**3. Цепное правило.** $L$ зависит от $w$ не напрямую, а через цепочку $w \to m \to z \to d \to L$. Правило дифференцирования сложной функции: производная цепочки равна **произведению** производных звеньев

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial d} \cdot \frac{\partial d}{\partial z} \cdot \frac{\partial z}{\partial m} \cdot \frac{\partial m}{\partial w} = 2d \cdot 1 \cdot 1 \cdot x = 2 \cdot 2 \cdot 2 = 8.$$

*Интуиция:* если $d$ растет в 2 раза быстрее, чем что-то, а это что-то – в 3 раза быстрее, чем $w$, то $d$ растет в $2 \cdot 3 = 6$ раз быстрее $w$. Скорости изменения перемножаются.

**4. Обратный проход.** Производные считают не для каждого веса отдельно, а одним проходом **от $L$ к входам**. Обозначим $\bar a = \dfrac{\partial L}{\partial a}$ – «градиент в узле $a$».

- Начало: $\bar L = \dfrac{\partial L}{\partial L} = 1$.
- Каждый узел получает градиент своего выхода и умножает его на **локальную производную** – производную своей простой операции по каждому входу:

| Операция | Локальные производные | Что передается назад |
|---|---|---|
| $c = a + b$ | $\partial c / \partial a = 1$, $\partial c / \partial b = 1$ | $\bar a \mathrel{+}= \bar c$, $\bar b \mathrel{+}= \bar c$ |
| $c = a \cdot b$ | $\partial c / \partial a = b$, $\partial c / \partial b = a$ | $\bar a \mathrel{+}= b\,\bar c$, $\bar b \mathrel{+}= a\,\bar c$ |
| $c = \mathrm{ReLU}(a)$ | 1 при $a > 0$, иначе 0 | $\bar a \mathrel{+}= [a > 0]\,\bar c$ |
| $c = \tanh a$ | $1 - \tanh^2 a = 1 - c^2$ | $\bar a \mathrel{+}= (1 - c^2)\,\bar c$ |

Для примера: $\bar d = 2d = 4$; вычитание и сложение передают 4 без изменений: $\bar z = 4$, $\bar b = 4$, $\bar m = 4$; умножение: $\bar w = x \cdot \bar m = 8$, $\bar x = w \cdot \bar m = 12$. Узлу не нужно знать весь граф – только свою операцию и пришедший градиент. Поэтому обратный проход стоит примерно столько же, сколько прямой (Гудфеллоу, раздел 6.5).

**5. Ветвление: градиенты складываются.** Если значение используется в нескольких местах, изменение этого значения влияет на $L$ по нескольким дорогам, и вклады **складываются**. Пример: $c = a \cdot a$ при $a = 3$. Оба входа умножения – одно и то же $a$: первый вход получает $a \cdot \bar c = 3$, второй – еще 3, итого $6 = 2a$, как и должно быть для $a^2$. В сети это повсюду: каждый вход слоя идет во все нейроны. Поэтому в коде градиенты **прибавляются** (`+=`), а не присваиваются.

**6. Порядок обхода.** Узел может передать градиент дальше только после того, как получил вклады от **всех** узлов, которые его используют. Такой порядок дает **топологическая сортировка**: выстроим узлы так, чтобы каждый стоял после всех своих «детей» (входов), и пройдем этот список с конца. Сортировку строит рекурсивный обход в глубину: сначала добавить в список всех детей узла, затем сам узел.

**7. Autograd в PyTorch.** Все это PyTorch делает сам. Тензор с флагом `requires_grad=True` запоминает каждую операцию над собой – строит граф. Вызов `L.backward()` проходит граф назад, результат лежит в поле `.grad` каждого листа: `w.grad` – это $\partial L / \partial w$. Три важные детали:
- градиенты **накапливаются** (`+=`, как в п. 5): повторный `backward()` прибавит новые градиенты к старым, поэтому перед каждым шагом их сбрасывают – `optimizer.zero_grad()`;
- внутри `with torch.no_grad():` граф не строится – так делают при оценке модели и при ручном изменении весов;
- `backward()` вызывают у **скаляра** – числа-потери.

**8. Цикл обучения и мини-пакеты.** Один шаг обучения в PyTorch – пять строк: `optimizer.zero_grad()` (сбросить градиенты) → `model(x)` (прямой проход) → `loss_fn(...)` (потеря) → `loss.backward()` (обратный проход) → `optimizer.step()` (шаг $w \leftarrow w - \eta\,\bar w$ для всех весов). Считать градиент по всем объектам сразу точно, но медленно; по одному объекту – быстро, но направление «шумит». Обычно берут **мини-пакет** из 32–256 случайных объектов. Один проход по всем обучающим объектам называется **эпохой**.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# «Две луны»: 1000 точек, 70 % – обучение, 30 % – тест (генерируются, скачивать ничего не нужно)
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=RANDOM_STATE)
Xtr_np, Xte_np, ytr_np, yte_np = train_test_split(X_np, y_np, test_size=0.3, random_state=RANDOM_STATE)
Xtr, Xte = torch.tensor(Xtr_np, dtype=torch.float32), torch.tensor(Xte_np, dtype=torch.float32)
ytr, yte = torch.tensor(ytr_np, dtype=torch.long), torch.tensor(yte_np, dtype=torch.long)
print(Xtr.shape, Xte.shape, '  ожидается torch.Size([700, 2]) torch.Size([300, 2])')

In [ ]:
# Учебный пример со слайдов
# 1) Скаляр: L = (w*x + b - y)^2
x_s, w_s, b_s, y_s = 2.0, 3.0, 1.0, 5.0

# 2) Сеть XOR из занятия Б1.1 с «испорченным» выходным весом: w = [1, -1.5] вместо [1, -2]
X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = torch.tensor([0., 1., 1., 0.])

def spoiled_params(dtype=torch.float32):
    """Свежие копии испорченных весов [W, c, w, b] с requires_grad=True."""
    W = torch.tensor([[1., 1.], [1., 1.]], dtype=dtype)
    c = torch.tensor([0., -1.], dtype=dtype)
    w = torch.tensor([1., -1.5], dtype=dtype)
    b = torch.tensor(0., dtype=dtype)
    return [p.requires_grad_() for p in (W, c, w, b)]

**Задание Г1. Класс `Value`: сложение и умножение.**

`Value` – обертка над числом, которая помнит, как это число получено. У каждого объекта пять полей:
- `data` – само число;
- `grad` – градиент $\partial L / \partial(\text{это число})$, изначально 0;
- `_prev` – множество «детей»: из каких `Value` получено;
- `_op` – название операции (для отладки);
- `_backward` – функция, которая передает градиент детям.

Допишите в методах `__add__` и `__mul__`:
1. значение результата `out` – сумму или произведение чисел `self.data` и `other.data`;
2. тело функции `_backward` по таблице из п. 4 теории.

*Как работает `_backward`.* Функция определена **внутри** метода, поэтому «помнит» переменные `self`, `other` и `out` из момента создания (это называется замыканием). Когда позже, при обратном проходе, в `out.grad` окажется градиент, вызов `out._backward()` прибавит нужные доли к `self.grad` и `other.grad`.

*Зачем строка `other = other if isinstance(other, Value) else Value(other)`.* Чтобы можно было писать `a + 3`, а не только `a + Value(3)`: обычное число превращается в `Value`. Методы с буквой `r` (`__radd__`, `__rmul__`) нужны для записи с числом слева: `2 * a`. Вычитание и минус выражены через `+` и `*`, поэтому их обратный проход получится автоматически.

*Проверка вручную.* Для $e = a \cdot b$ при $a = 2$, $b = -3$ и $\bar e = 1$: $\bar a = b \cdot \bar e = -3$, $\bar b = a \cdot \bar e = 2$.

In [ ]:
class Value:
    """Число, которое помнит, откуда оно взялось."""
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0                   # dL/d(это число), заполняется обратным проходом
        self._backward = lambda: None     # как передать градиент детям
        self._prev = set(_children)       # из каких Value получено
        self._op = _op                    # какой операцией

    def __repr__(self):
        return f'Value(data={self.data:.4g}, grad={self.grad:.4g})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(..., (self, other), '+')  # <-- сумма чисел self.data и other.data
        def _backward():
            self.grad += ...   # <-- локальная производная суммы по слагаемому равна 1
            other.grad += ...  # <-- то же для второго слагаемого
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(..., (self, other), '*')  # <-- произведение чисел
        def _backward():
            self.grad += ...   # <-- d(a*b)/da = b, умножить на out.grad
            other.grad += ...  # <-- d(a*b)/db = a, умножить на out.grad
        out._backward = _backward
        return out

    # Готовые методы: все выражены через + и *
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return (-self) + other
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other


a, b = Value(2.0), Value(-3.0)
e = a * b
print(e.data, '  ожидается -6.0')
e.grad = 1.0
e._backward()
print(a.grad, b.grad, '  ожидается -3.0 2.0')

a, b = Value(2.0), Value(-3.0)
f = a + b
f.grad = 1.0
f._backward()
print(f.data, a.grad, b.grad, '  ожидается -1.0 1.0 1.0')
print((Value(5.0) - 3).data, (2 * Value(4.0)).data, '  ожидается 2.0 8.0')

**Вопрос Г1.** а) Что хранит поле `grad` у числа `a`? б) Почему в `_backward` стоит `+=`, а не `=`? Приведите выражение, где `=` даст неверный ответ.

*Ваш ответ:*

**Задание Г2. Активации и полный обратный проход.**

1. `relu(self)` – новая операция с одним входом. Значение $\max(0, x)$; локальная производная равна 1 при $x > 0$ и 0 иначе. В Python `True` ведет себя как 1, а `False` как 0, поэтому удобно писать `(out.data > 0) * out.grad`.
2. `tanh(self)` – значение $t = \tanh x$ уже посчитано. Его производная: $\tanh'(x) = 1 - \tanh^2 x = 1 - t^2$. (Вывод: $\tanh x = \dfrac{\sinh x}{\cosh x}$, по правилу частного $\tanh' = \dfrac{\cosh^2 x - \sinh^2 x}{\cosh^2 x} = 1 - \tanh^2 x$.) Производная выражается через уже известное значение $t$ – поэтому пересчитывать экспоненты не нужно.
3. `backward(self)` – весь обратный проход от этого числа (обычно – от потери $L$):
   - построить список `topo` (п. 6 теории): рекурсивная функция `build(v)` сначала вызывает себя для всех детей `v`, затем добавляет `v`; множество `visited` не дает добавить узел дважды, если к нему ведут несколько дорог;
   - положить `self.grad = 1.0` – это $\partial L / \partial L$;
   - пройти `topo` **с конца** (`reversed`) и у каждого узла вызвать `_backward()`.

*Пример порядка.* Для $L = (w x + b - y)^2$ список получится примерно таким: $w, x, m, b, z, y, \dots, d, L$ – входы в начале, $L$ в конце. С конца: сначала $L$ отдает градиент $d$, затем $d$ отдает $z$ и так далее, и каждый узел к моменту своей очереди уже собрал все вклады.

Функции дописываются к классу строкой `Value.relu = relu` – после нее у всех объектов `Value` появляется метод `.relu()`.

In [ ]:
def relu(self):
    out = Value(..., (self,), 'relu')  # <-- max(0, x)
    def _backward():
        self.grad += ...  # <-- производная ReLU (используйте out.data > 0) умножить на out.grad
    out._backward = _backward
    return out

def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += ...  # <-- (1 - t^2) умножить на out.grad
    out._backward = _backward
    return out

def backward(self):
    topo, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                ...           # <-- рекурсивно добавьте в порядок ребенка
            topo.append(v)
    build(self)
    self.grad = ...           # <-- dL/dL
    for v in reversed(topo):
        ...                   # <-- передайте градиент детям узла v

Value.relu, Value.tanh, Value.backward = relu, tanh, backward


x = Value(0.5)
out = (x * 2).tanh()
out.backward()
print(round(out.data, 4), round(x.grad, 4), '  ожидается 0.7616 0.8399')

r = Value(-1.0)
s = r.relu()
s.backward()
print(s.data, r.grad, '  ожидается 0.0 0.0')

a = Value(3.0)
sq = a * a
sq.backward()
print(a.grad, '  ожидается 6.0 (a используется дважды – градиенты складываются)')

**Вопрос Г2.** а) Зачем нужен топологический порядок? Что сломается, если вызывать `_backward` в произвольном порядке? б) Почему градиент ReLU при $x = -1$ равен нулю и чем это опасно для обучения?

*Ваш ответ:*

**Задание Г3. Выражение со слайда и один шаг спуска.**

- `slide_loss(x, w, b, y)` строит из объектов `Value` потерю $L = (w x + b - y)^2$. Квадрат запишите как `d * d` – заодно проверите, что градиенты на ветвлении складываются (п. 5 теории).
- `sgd_step(params, lr)` – шаг градиентного спуска для списка `Value`: у каждого параметра `data -= lr * grad`.

*Ожидаемый разбор (как на доске).*

| Узел | Прямой проход | Градиент (обратный проход) |
|---|---|---|
| $L = d \cdot d$ | 4 | 1 |
| $d = z - y$ | 2 | $d + d = 4$ |
| $z = m + b$ | 7 | 4 |
| $b$ | 1 | 4 |
| $m = w \cdot x$ | 6 | 4 |
| $w$ | 3 | $x \cdot 4 = 8$ |
| $x$ | 2 | $w \cdot 4 = 12$ |

Шаг с $\eta = 0{,}01$: $w = 3 - 0{,}01 \cdot 8 = 2{,}92$, $b = 1 - 0{,}01 \cdot 4 = 0{,}96$. Новая потеря: $z = 2{,}92 \cdot 2 + 0{,}96 = 6{,}8$, $d = 1{,}8$, $L = 3{,}24$ – меньше, чем 4, значит, шаг сделан в правильную сторону. Обучаем только $w$ и $b$: $x$ и $y$ – данные, их менять нельзя.

In [ ]:
def slide_loss(x, w, b, y):
    d = ...        # <-- w*x + b - y из операций Value
    return ...     # <-- квадрат через умножение

def sgd_step(params, lr):
    for p in params:
        ...        # <-- шаг против градиента


x, w, b, y = Value(x_s), Value(w_s), Value(b_s), Value(y_s)
L = slide_loss(x, w, b, y)
L.backward()
print(L.data, '  ожидается 4.0')
print(w.grad, b.grad, x.grad, '  ожидается 8.0 4.0 12.0')

sgd_step([w, b], lr=0.01)
print(round(w.data, 4), round(b.data, 4), '  ожидается 2.92 0.96')
print(round(slide_loss(x, w, b, y).data, 4), '  ожидается 3.24 (потеря уменьшилась)')

In [ ]:
# Готовая ячейка: сверка с autograd PyTorch
xt, wt, bt = (torch.tensor(v, requires_grad=True) for v in (x_s, w_s, b_s))
Lt = (wt * xt + bt - y_s) ** 2
Lt.backward()
print(Lt.item(), wt.grad.item(), bt.grad.item(), xt.grad.item(), '  ожидается 4.0 8.0 4.0 12.0')

**Вопрос Г3.** а) Градиент по $x$ получился 12, хотя $x$ мы не обучаем. Где в сети такой градиент нужен? б) Сравните шаг `sgd_step` с градиентным спуском для логистической регрессии из 9 класса. Что общего и что нового?

*Ваш ответ:*

**Задание Г4. Сеть XOR на `torch` и проверка градиентов.**

Теперь работаем с тензорами, и обратный проход делает autograd.

- `xor_forward(W, c, w, b)` – выход сети $\hat y = \mathrm{ReLU}(XW + c)\,w + b$ на четырех точках `X_xor` (теперь можно `torch.relu`). Метод `.to(W.dtype)` приводит данные к типу весов: в этом задании веса хранятся в `float64` (почему – ниже).
- `xor_loss(W, c, w, b)` – среднеквадратичная ошибка $L = \frac14 \sum_i (\hat y_i - y_i)^2$.
- `numerical_grad(f, t, eps)` – градиент функции `f()` по каждому элементу тензора `t` **центральной разностью**

$$\frac{\partial L}{\partial t_i} \approx \frac{f(t_i + \varepsilon) - f(t_i - \varepsilon)}{2\varepsilon}.$$

  Элемент временно сдвигается на месте и возвращается обратно. Все это делается внутри `torch.no_grad()` и через `t.data` – ручное изменение весов не должно попадать в граф вычислений.

*Обратный проход для сети XOR в матричном виде (проверьте числа на бумаге).* С испорченным весом $w = (1; -1{,}5)$ скрытый слой прежний: $H = \begin{pmatrix}0&0\\1&0\\1&0\\2&1\end{pmatrix}$, выход $\hat y = Hw = (0;\ 1;\ 1;\ 0{,}5)$, ошибается только точка $(1, 1)$, потеря $L = 0{,}5^2 / 4 = 0{,}0625$.

1. Градиент по выходу: $g = \dfrac{\partial L}{\partial \hat y} = \dfrac{2(\hat y - y)}{4} = (0;\ 0;\ 0;\ 0{,}25)$.
2. Выходной слой – как у линейной регрессии: $\dfrac{\partial L}{\partial w} = H^\top g = (0{,}5;\ 0{,}25)$, $\dfrac{\partial L}{\partial b} = \sum_i g_i = 0{,}25$.
3. Назад в скрытый слой: $\dfrac{\partial L}{\partial H} = g\,w^\top$; ненулевая только четвертая строка $0{,}25 \cdot (1; -1{,}5) = (0{,}25;\ -0{,}375)$.
4. Через ReLU: вход четвертой точки $z_4 = (2, 1)$ положителен, градиент проходит без изменений.
5. Параметры скрытого слоя: $\dfrac{\partial L}{\partial c}$ – сумма строк, то есть $(0{,}25;\ -0{,}375)$; $\dfrac{\partial L}{\partial W} = X^\top \dfrac{\partial L}{\partial Z}$, и так как $x_4 = (1, 1)$, обе строки равны $(0{,}25;\ -0{,}375)$.

*Почему `float64` и почему разница не ноль.* В `float32` около 7 значащих цифр: при $\varepsilon = 10^{-4}$ разность $f(t + \varepsilon) - f(t - \varepsilon)$ теряет почти всю точность. В `float64` – около 16 цифр, запаса хватает. Небольшое расхождение с autograd все равно останется: центральная разность – приближение с ошибкой порядка $\varepsilon^2$, а у точек 2 и 3 вход второго скрытого нейрона равен ровно нулю – это излом ReLU, где производной нет, и autograd по соглашению берет 0.

In [ ]:
def xor_forward(W, c, w, b):
    H = ...        # <-- ReLU(X_xor @ W + c)
    return ...     # <-- H @ w + b

def xor_loss(W, c, w, b):
    return ...     # <-- среднее квадратов ошибок

def numerical_grad(f, t, eps=1e-4):
    grad = torch.zeros_like(t)
    with torch.no_grad():
        flat, gflat = t.data.view(-1), grad.view(-1)
        for i in range(flat.numel()):
            old = flat[i].item()
            flat[i] = old + eps
            f_plus = f().item()
            flat[i] = ...              # <-- сдвиг в другую сторону
            f_minus = f().item()
            flat[i] = old
            gflat[i] = ...             # <-- центральная разность
    return grad


params = spoiled_params(torch.float64)
W, c, w, b = params
L = xor_loss(*params)
L.backward()
print(show(xor_forward(*params)), '  ожидается [0.0, 1.0, 1.0, 0.5]')
print(show(L, 4), '  ожидается 0.0625')
print(show(w.grad), show(b.grad), '  ожидается [0.5, 0.25] 0.25')
print(show(c.grad), '  ожидается [0.25, -0.375]')
print(show(W.grad), '  ожидается [[0.25, -0.375], [0.25, -0.375]]')

f = lambda: xor_loss(*params)
max_diff = max((numerical_grad(f, p) - p.grad).abs().max().item() for p in params)
print(f'{max_diff:.1e}', '  ожидается < 1e-03')

**Вопрос Г4.** а) Почему градиенты по $c$ и $W$ получились только от четвертой точки? б) Сколько вычислений потери нужно конечным разностям для сети 784 → 128 → 10? Почему их используют только для проверки? в) Почему разница с autograd не ноль, хотя обе величины «правильные»?

*Ваш ответ:*

**Задание Г5. Цикл обучения на PyTorch.**

`train_xor(params, lr, steps)` обучает сеть XOR и возвращает список значений потери на каждом шаге. Вместо ручного `sgd_step` используем готовый оптимизатор `torch.optim.SGD(params, lr=lr)`: он получает список параметров и умеет две вещи – `zero_grad()` (сбросить градиенты всех параметров) и `step()` (сделать для каждого параметра $p \leftarrow p - \eta \cdot p.\mathrm{grad}$).

Каждый шаг цикла – пять действий (п. 8 теории):
1. обнулить градиенты – иначе они накопятся с прошлых шагов;
2. посчитать потерю – прямой проход строит граф;
3. `loss.backward()` – обратный проход заполняет `.grad` всех параметров;
4. `optimizer.step()` – шаг против градиента;
5. записать `loss.item()` – `.item()` превращает тензор-скаляр в обычное число Python, чтобы список не хранил графы вычислений.

*Ожидаемый первый шаг (посчитайте по градиентам из Г4).* С $\eta = 0{,}1$: $w = (1 - 0{,}05;\ -1{,}5 - 0{,}025) = (0{,}95;\ -1{,}525)$, $b = 0 - 0{,}025 = -0{,}025$; одновременно поправляются $W$ и $c$, и потеря падает с $0{,}0625$ до $0{,}031$. За 100 шагов она опускается ниже $0{,}01$.

In [ ]:
def train_xor(params, lr=0.1, steps=100):
    optimizer = torch.optim.SGD(params, lr=lr)
    history = []
    for step in range(steps):
        ...                 # <-- обнулите градиенты оптимизатора
        loss = ...          # <-- потеря xor_loss на текущих params
        ...                 # <-- обратный проход
        ...                 # <-- шаг оптимизатора
        history.append(loss.item())
    return history


params = spoiled_params()
hist = train_xor(params, lr=0.1, steps=1)
print(show(hist[0], 4), '  ожидается 0.0625 (потеря до шага)')
print(show(params[2]), show(params[3]), '  ожидается [0.95, -1.525] -0.025')
print(show(xor_loss(*params)), '  ожидается 0.031')

params = spoiled_params()
hist = train_xor(params, lr=0.1, steps=100)
print(hist[-1] < 0.01, '  ожидается True')
print(show(torch.round(xor_forward(*params))), '  ожидается [0.0, 1.0, 1.0, 0.0]')

In [ ]:
# Готовая ячейка: кривая обучения XOR
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(hist, color='#8F8260', lw=2)
ax.set_yscale('log'); ax.set_xlabel('шаг'); ax.set_ylabel('потеря (MSE)'); ax.set_title('Обучение сети XOR, lr = 0.1')
fig.tight_layout(); fig.savefig('xor_loss.png', dpi=120); plt.close(fig)
display(Image('xor_loss.png'))

**Вопрос Г5.** а) Что произойдет, если убрать строку с `zero_grad`? Проверьте на 20 шагах и объясните через накопление градиентов. б) Замените `lr=0.1` на `lr=0.2` и `lr=0.5`. Что видно на кривой?

*Ваш ответ:*

**Задание Г6. Обучение по мини-пакетам: `train_epoch` и `evaluate`.**

Эти две функции будут использоваться во всех следующих работах, поэтому напишите их аккуратно.

- `train_epoch(model, X, y, loss_fn, optimizer, batch_size, generator)` – одна эпоха (п. 8 теории):
  1. `model.train()` – режим обучения (у некоторых слоев, например прореживания в Блоке 2, поведение при обучении и при оценке разное);
  2. `torch.randperm(len(X))` – случайная перестановка номеров объектов, чтобы пакеты каждый раз были разными; `generator` с фиксированным сидом делает перемешивание воспроизводимым;
  3. для каждого пакета – пять строк цикла обучения;
  4. средняя потеря эпохи: потеря пакета – это среднее по его объектам, поэтому умножаем ее на размер пакета, складываем и делим на число объектов (последний пакет может быть меньше остальных).
- `evaluate(model, X, y, loss_fn)` – потеря и доля верных ответов без обучения: `model.eval()` и `torch.no_grad()` (граф не нужен – веса не меняются). Предсказанный класс – номер наибольшего логита: `logits.argmax(dim=1)`.

*Про функцию потерь.* У модели два выхода – логиты двух классов, а `nn.CrossEntropyLoss` принимает именно логиты и номера верных классов: softmax и $-\ln p_y$ она считает внутри себя (п. 5 теории части 1С). Такая схема подходит для любого числа классов – в следующих работах их будет 10. До обучения сеть почти ничего не знает: вероятности близки к $0{,}5$, потеря близка к $-\ln 0{,}5 = \ln 2 \approx 0{,}69$.

*Сколько шагов.* При 700 объектах и пакетах по 32 за эпоху делается $\lceil 700 / 32 \rceil = 22$ шага, за 100 эпох – 2200.

In [ ]:
def train_epoch(model, X, y, loss_fn, optimizer, batch_size=32, generator=None):
    model.train()
    perm = torch.randperm(len(X), generator=generator)
    total = 0.0
    for start in range(0, len(X), batch_size):
        idx = perm[start:start + batch_size]
        xb, yb = X[idx], y[idx]
        ...                          # <-- обнулите градиенты
        loss = ...                   # <-- потеря модели на пакете
        ...                          # <-- обратный проход
        ...                          # <-- шаг оптимизатора
        total += loss.item() * len(idx)
    return total / len(X)

def evaluate(model, X, y, loss_fn):
    model.eval()
    with torch.no_grad():
        logits = ...                 # <-- выход модели на всех объектах
        loss = loss_fn(logits, y).item()
        acc = ...                    # <-- доля совпадений argmax(dim=1) с y, число float
    return loss, acc


torch.manual_seed(RANDOM_STATE)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
g = torch.Generator().manual_seed(RANDOM_STATE)

loss0, acc0 = evaluate(model, Xte, yte, loss_fn)
print(round(loss0, 2), '  ожидается порядка 0.7 (случайные веса, ln 2 = 0.69)')

In [ ]:
# Готовая ячейка обучения: 100 эпох, пакеты по 32 объекта
train_hist, test_hist = [], []
for epoch in range(100):
    train_hist.append(train_epoch(model, Xtr, ytr, loss_fn, optimizer, batch_size=32, generator=g))
    test_hist.append(evaluate(model, Xte, yte, loss_fn)[0])
test_loss, test_acc = evaluate(model, Xte, yte, loss_fn)
print(train_hist[-1] < 0.15, '  ожидается True (потеря на обучении ниже 0.15)')
print(round(test_acc, 3), '  ожидается не ниже 0.93')

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(train_hist, color='#8F8260', lw=2, label='обучение')
ax.plot(test_hist, color='#6E6E78', lw=2, ls='--', label='тест')
ax.set_xlabel('эпоха'); ax.set_ylabel('кросс-энтропия'); ax.legend(); ax.set_title('«Две луны», пакеты по 32')
fig.tight_layout(); fig.savefig('moons_loss.png', dpi=120); plt.close(fig)
display(Image('moons_loss.png'))

**Вопрос Г6.** а) Сколько шагов оптимизатора делается за одну эпоху при 700 объектах и пакетах по 32? б) Зачем в `evaluate` нужен `torch.no_grad()`? в) Сравните кривые обучения и теста. Есть ли переобучение? Сравните с деревом решений без ограничения глубины из 9 класса.

*Ваш ответ:*

### Задание повышенной сложности (к части 2Г)

1. **Микро-библиотека.** На классе `Value` напишите классы `Neuron`, `Layer`, `MLP` (как в micrograd) и обучите сеть 2 → 4 → 1 с `tanh` на четырех точках XOR своим циклом: обнуление `grad`, прямой проход, `backward`, шаг. Сравните число строк с PyTorch-версией.
2. **Без `zero_grad`.** В `train_xor` уберите обнуление градиентов и нарисуйте кривую потери для `lr=0.1` и `lr=0.01`. Объясните, почему это похоже на увеличение шага.
3. **Мертвые нейроны.** Обучите XOR с `lr=0.5` и после первого шага выведите вход скрытого слоя $XW + c$ на всех четырех точках. Докажите, что после этого градиенты по $W$ и $c$ навсегда нулевые.
4. **Размер пакета.** Обучите модель из Г6 с `batch_size` 1, 32 и 700 одинаковое число **шагов** (например, 2200). Сравните время, кривые потерь и точность на тесте.

In [ ]:
# Место для задания повышенной сложности